# Task 1: universal denoising autoencoder (Kaggle run)

Before running:
1. Notebook settings: turn the GPU accelerator and Internet on.
2. Add your Weights & Biases key as a secret named `WANDB_API_KEY` and attach it to this notebook.
3. If the GitHub repository is private, also add a GitHub token as a secret named `GITHUB_TOKEN`.

This notebook only launches the scripts in the repository; all logic lives in `src/`.

In [ ]:
# 1. Secrets -> environment variables
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
os.environ["WANDB_PROJECT"] = "genai-a1"

REPO = "aleenababar04/genai-assignment-1"
try:
    token = secrets.get_secret("GITHUB_TOKEN")   # only needed for a private repo
    REPO_URL = f"https://{token}@github.com/{REPO}.git"
except Exception:
    REPO_URL = f"https://github.com/{REPO}.git"


In [ ]:
# 2. Get the code and install the few packages Kaggle does not ship
%cd /kaggle/working
!rm -rf genai-assignment-1
!git clone -q $REPO_URL genai-assignment-1
%cd genai-assignment-1
!git log --oneline -1
!pip install -q optuna wandb onnx onnxruntime pytorch-msssim python-dotenv

import torch
print("torch", torch.__version__, "| GPU available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Turn the GPU on in the notebook settings."


In [ ]:
# 3. Download Oxford-IIIT Pet and build the 128x128 cache.
#    The split is checked against manifests/pet_split.json from the repo.
!python -m src.data.prepare_pet
!python -m pytest -q


In [ ]:
# 4. Optuna search (30 trials x 10 epochs; weak trials are pruned early).
#    Writes optuna_studies/task1_udae.db and configs/task1_udae_best.yaml.
!python -m src.training.train_udae --mode optuna


In [ ]:
# 5. Full training with the best configuration -> checkpoints/task1_udae.pt
!python -m src.training.train_udae --mode final


In [ ]:
# 6. Ablation: the same configuration with one limited skip connection at 16x16
#    -> checkpoints/task1_udae_skip.pt
!python -m src.training.train_udae --mode final --skip


In [ ]:
# 7. Test-set evaluation (tables and figures) for the main model and the ablation
!python -m src.evaluation.eval_udae
!python -m src.evaluation.eval_udae --name task1_udae_skip


In [ ]:
# 8. ONNX export of the main model, verified against PyTorch
!python -m src.export.export_onnx --task task1


In [ ]:
# 9. Collect everything that must go back into the repository.
#    Download task1_outputs.zip from the notebook's Output panel.
!mkdir -p /kaggle/working/task1_outputs
!cp -r optuna_studies configs/task1_udae_best.yaml checkpoints models report/results report/figures /kaggle/working/task1_outputs/
%cd /kaggle/working
!zip -qr task1_outputs.zip task1_outputs
!ls -lh task1_outputs.zip
